In [1]:
# Show CLI help
CONTAINER=$SHARED/alphafold3/images/openfold3_3cfe483-euler1.sif
TRITON_CACHE_DIR=$TMPDIR/_triton_cache_dir
mkdir -p $TRITON_CACHE_DIR
singularity exec --nv --writable-tmpfs \
  --env OPENFOLD_CACHE=/root/.openfold3/ \
  --env TRITON_CACHE_DIR=$TRITON_CACHE_DIR \
  --bind $TRITON_CACHE_DIR \
  $CONTAINER \
  sh -c 'run_openfold predict --help'

Usage: run_openfold predict [OPTIONS]

  Perform inference on a set of queries defined in the query_json.

Options:
  --query_json FILE               Json containing the queries for prediction.
                                  [required]
  --inference_ckpt_path PATH      Path for model checkpoint to be used for
                                  inference. If not specified, will attempt to
                                  find or download parameters to
                                  $OPENFOLD_CACHE [default: ~/.openfold3/]
  --num_diffusion_samples INTEGER
                                  Number of diffusion samples to generate for
                                  each query.
  --num_model_seeds INTEGER       Number of model seeds to use for each query.
  --runner_yaml FILE              Yaml that specifies model and dataset
                                  parameters, see examples/runner.yml
  --use_msa_server BOOLEAN        Use ColabFold MSA server to perform
                  

In [3]:
# Run with directory overlay (--overlay) to build pytorch extensions: https://github.com/aqlaboratory/openfold-3/issues/17
rm -rf openfold3_predictions
rm -rf $TMPDIR/_overlay
rm -rf $TMPDIR/_triton_cache_dir
mkdir -p openfold3_predictions/
mkdir -p $TMPDIR/_overlay
mkdir -p $TMPDIR/_triton_cache_dir
mkdir -p $TMPDIR/_xdg_cache_home
unset XDG_CACHE_HOME
singularity exec --nv \
  --env OPENFOLD_CACHE=/root/.openfold3/ \
  --env TRITON_CACHE_DIR=$TMPDIR/_triton_cache_dir \
  --bind $TMPDIR/_triton_cache_dir \
  --env XDG_CACHE_HOME=$TMPDIR/_xdg_cache_home \
  --bind $TMPDIR/_xdg_cache_home \
  --bind openfold3_jsons:/root/openfold3_jsons \
  --bind openfold3_predictions:/root/openfold3_predictions \
  --overlay $TMPDIR/_overlay \
  $CONTAINER \
  sh -c 'python3 -c "import deepspeed; deepspeed.ops.op_builder.EvoformerAttnBuilder().load()"; run_openfold predict --use_templates False --query_json /root/openfold3_jsons/ubiquitin.json --output_dir /root/openfold3_predictions/ubiquitin'

/opt/conda/envs/openfold3/lib/python3.12/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
/opt/openfold3/openfold3/core/utils/checkpoint_loading_utils.py:49: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add

In [ ]:
# Output is there..
ls -lh openfold3_predictions/ubiquitin

total 84K
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 4.4K Apr 27 15:28 experiment_config.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 1.3K Apr 27 15:29 inference_query_set.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  59K Apr 27 15:28 model_config.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  273 Apr 27 15:29 summary.txt
drwxr-sr-x 3 jjaenes biol-imsb-beltrao 4.0K Apr 27 15:29 ubiquitin


In [ ]:
# Predicted structures are there...
ls -lh openfold3_predictions/ubiquitin/ubiquitin/seed_42

total 1.4M
-rw-r--r-- 1 jjaenes biol-imsb-beltrao   39 Apr 27 15:29 timing.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  316 Apr 27 15:29 ubiquitin_seed_42_sample_1_confidences_aggregated.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 197K Apr 27 15:29 ubiquitin_seed_42_sample_1_confidences.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  62K Apr 27 15:29 ubiquitin_seed_42_sample_1_model.cif
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  319 Apr 27 15:29 ubiquitin_seed_42_sample_2_confidences_aggregated.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 197K Apr 27 15:29 ubiquitin_seed_42_sample_2_confidences.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  63K Apr 27 15:29 ubiquitin_seed_42_sample_2_model.cif
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  319 Apr 27 15:29 ubiquitin_seed_42_sample_3_confidences_aggregated.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 197K Apr 27 15:29 ubiquitin_seed_42_sample_3_confidences.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao  62K Apr 27 15:29 ubiquitin_seed_42_sample_3_model.cif